In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "cell-0",
   "metadata": {},
   "source": [
    "# 🏭 Industrial Predictive Maintenance & Fault Diagnostic Engine\n",
    "## Industrial Predictive Maintenance & Fault Diagnostic Engine\n",
    "\n",
    "---\n",
    "\n",
    "# Phase 1: Problem & Data Framing\n",
    "\n",
    "This is the first of the six **CRISP-DM** phases. Its goal is to lay a **solid foundation** before writing any modeling code, because every decision we make later is built on it.\n",
    "\n",
    "## 🎯 What we build\n",
    "\n",
    "A system that predicts machine failure **before it happens** from live sensor readings, then **diagnoses the failure type**. These are **two tasks** built on the same data:\n",
    "\n",
    "| Task | Type | Target |\n",
    "|---------|-------|-------|\n",
    "| Will the machine fail? | Binary classification | `Machine failure` = 0/1 |\n",
    "| What failure type? | Multi-class classification | `Failure Type` = TWF/HDF/PWF/OSF/RNF |\n",
    "\n",
    "## 📦 Dataset\n",
    "\n",
    "We use the **AI4I 2020 Predictive Maintenance Dataset** from Kaggle, an academic/industrial benchmark simulating a milling machine with 10,000 records and the following sensors:\n",
    "\n",
    "- `Air temperature [K]` — ambient air temperature\n",
    "- `Process temperature [K]` — manufacturing process temperature\n",
    "- `Rotational speed [rpm]` — rotational speed\n",
    "- `Torque [Nm]` — torque\n",
    "- `Tool wear [min]` — cutting tool wear\n",
    "\n",
    "> 💡 **Kaggle path:** `/kaggle/input/ai4i-predictive-maintenance-dataset/ai4i2020.csv`\n",
    "\n",
    "## 🛠️ Quality rules we commit to throughout the project\n",
    "\n",
    "1. **No magic numbers:** every constant is defined once at the top of the notebook and used by name.\n",
    "2. **Documented functions:** every piece of logic is wrapped in a function with a clear name + docstring + type hints.\n",
    "3. **Data quality checks:** we verify consistency before trusting the data.\n",
    "4. **Reproducibility:** a fixed random seed `RANDOM_STATE`.\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-1",
   "metadata": {},
   "source": [
    "## 1.1 — Setup: imports and central constants\n",
    "\n",
    "**What did we use and why?**\n",
    "\n",
    "| Element | Reason |\n",
    "|--------|-------|\n",
    "| `pathlib.Path` | Handle paths as objects instead of strings — more precise and safer across platforms |\n",
    "| `RANDOM_STATE = 42` | Fix the random seed so all results are **fully reproducible** |\n",
    "| Column constants | Single Source of Truth — if a column name changes, we edit it in one place only |\n",
    "\n",
    "> ⚠️ Note: the column names here contain **spaces and square brackets**, so they cannot be accessed via `df.col_name` but via `df[\"column name\"]`.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-2",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# Phase 1 — Setup and constants\n",
    "# ============================================================\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "\n",
    "# ─── General constants ────────────────────────────────────\n",
    "RANDOM_STATE = 42          # random seed: ensures reproducibility\n",
    "np.random.seed(RANDOM_STATE)\n",
    "\n",
    "# ─── Data paths (Kaggle) ─────────────────────────────────\n",
    "DATA_DIR  = Path(\"/kaggle/input/ai4i-predictive-maintenance-dataset\")\n",
    "DATA_FILE = DATA_DIR / \"ai4i2020.csv\"\n",
    "\n",
    "# ─── Column groups ────────────────────────────────────────\n",
    "# Identifier columns (carry no physical information for the model)\n",
    "ID_COLUMNS = [\"UDI\", \"Product ID\"]\n",
    "\n",
    "# Sensor columns = physical features\n",
    "SENSOR_COLUMNS = [\n",
    "    \"Air temperature [K]\",\n",
    "    \"Process temperature [K]\",\n",
    "    \"Rotational speed [rpm]\",\n",
    "    \"Torque [Nm]\",\n",
    "    \"Tool wear [min]\",\n",
    "]\n",
    "\n",
    "# Binary target\n",
    "BINARY_TARGET = \"Machine failure\"\n",
    "\n",
    "# Failure-type columns (encoded as 0/1 flags inside the file)\n",
    "FAILURE_TYPE_COLUMNS = [\"TWF\", \"HDF\", \"PWF\", \"OSF\", \"RNF\"]\n",
    "\n",
    "# Column we will derive for the multi-class task\n",
    "MULTI_CLASS_TARGET = \"Failure Type\"\n",
    "\n",
    "print(\"Constants ready ✅\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-3",
   "metadata": {},
   "source": [
    "## 1.2 — Load data function\n",
    "\n",
    "**Why wrap loading in a function instead of writing it directly?**\n",
    "\n",
    "1. **Reuse:** we will call it in later phases without repetition.\n",
    "2. **Error handling:** we check the file exists and give a clear message instead of a cryptic crash.\n",
    "3. **Documentation:** the docstring explains inputs and outputs to anyone reading the code later.\n",
    "\n",
    "> Note: `Type` here is the **product quality** (Low/Medium/High), an important categorical variable we will study in Phase 3.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-4",
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_data(path: Path) -> pd.DataFrame:\n",
    "    \"\"\"\n",
    "    Load the predictive maintenance dataset from a CSV file.\n",
    "\n",
    "    Parameters\n",
    "    ----------\n",
    "    path : Path\n",
    "        Full path to ai4i2020.csv.\n",
    "\n",
    "    Returns\n",
    "    -------\n",
    "    pd.DataFrame\n",
    "        Raw data as-is (no preprocessing).\n",
    "\n",
    "    Raises\n",
    "    ------\n",
    "    FileNotFoundError\n",
    "        If the file does not exist at the given path.\n",
    "    \"\"\"\n",
    "    if not path.exists():\n",
    "        raise FileNotFoundError(\n",
    "            f\"File not found: {path}\\n\"\n",
    "            \"Make sure the 'AI4I 2020 Predictive Maintenance' dataset is added to your Kaggle notebook.\"\n",
    "        )\n",
    "    return pd.read_csv(path)\n",
    "\n",
    "# ─── Load the data ────────────────────────────────────────\n",
    "df = load_data(DATA_FILE)\n",
    "print(f\"Loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-5",
   "metadata": {},
   "source": [
    "## 1.3 — Build both targets (binary and multi-class)\n",
    "\n",
    "Surprise: the dataset does **not** contain a \"Failure Type\" column directly! Instead it contains:\n",
    "- A binary column `Machine failure` (did a failure occur?).\n",
    "- Five flag columns `TWF, HDF, PWF, OSF, RNF` (failure type, encoded as 0/1 per type).\n",
    "\n",
    "So we **derive** the `Failure Type` column ourselves. But during inspection we discovered a **documented internal inconsistency** in this data:\n",
    "\n",
    "> ⚠️ The failure-type flags are generated from **physical rules** (e.g. temp diff < 8.6K and speed < 1380 rpm → HDF), while the `Machine failure` column was set independently. Result: some rows carry a failure type (e.g. `HDF=1`) but `Machine failure=0`.\n",
    "\n",
    "**The policy we adopt (professionally correct):**\n",
    "\n",
    "1. **Binary target** stays `Machine failure` — the official truth for overall failure.\n",
    "2. **Failure type** is derived only from rows that actually failed (`Machine failure == 1`).\n",
    "3. **Co-occurring failures** are joined with `+` (e.g. `TWF+HDF`).\n",
    "4. We document the mismatch size instead of crashing with `assert`.\n",
    "\n",
    "**Meaning of the failure types:**\n",
    "| Code | Meaning |\n",
    "|-------|--------|\n",
    "| TWF | Tool Wear Failure |\n",
    "| HDF | Heat Dissipation Failure |\n",
    "| PWF | Power Failure |\n",
    "| OSF | Overstrain Failure |\n",
    "| RNF | Random Failure |\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-6",
   "metadata": {},
   "outputs": [],
   "source": [
    "def build_targets(df: pd.DataFrame) -> pd.DataFrame:\n",
    "    \"\"\"\n",
    "    Derive the multi-class target while reconciling the known\n",
    "    inconsistency in the AI4I dataset.\n",
    "\n",
    "    Important note: in the original AI4I data, the 'Machine failure' column\n",
    "    does not exactly match the failure-type flags; some rows carry a failure\n",
    "    type without the overall failure flag. Here we treat 'Machine failure'\n",
    "    as the ground truth for overall failure, and derive the failure type\n",
    "    only from rows that actually failed.\n",
    "    \"\"\"\n",
    "    df = df.copy()\n",
    "    flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)\n",
    "\n",
    "    # ── Diagnose the mismatch (understand before fixing) ─────────\n",
    "    type_without_failure = (flag_sum >= 1) & (df[BINARY_TARGET] == 0)\n",
    "    failure_without_type = (flag_sum == 0) & (df[BINARY_TARGET] == 1)\n",
    "    print(f\"🔎 Rows with a failure type but no Machine failure: {type_without_failure.sum()}\")\n",
    "    print(f\"🔎 Rows with Machine failure but no failure type : {failure_without_type.sum()}\")\n",
    "\n",
    "    # ── Policy ─────────────────────────────────\n",
    "    # 1) Binary target stays: 'Machine failure' (ground truth).\n",
    "    # 2) Failure type is derived only from failed rows (Machine failure == 1).\n",
    "    # 3) Co-occurring failures are joined with '+' (e.g. TWF+HDF).\n",
    "    df[MULTI_CLASS_TARGET] = \"No Failure\"\n",
    "    failed_mask = df[BINARY_TARGET] == 1\n",
    "    df.loc[failed_mask, MULTI_CLASS_TARGET] = (\n",
    "        df.loc[failed_mask, FAILURE_TYPE_COLUMNS]\n",
    "        .apply(lambda row: \"+\".join(row.index[row == 1]) or \"Unknown\", axis=1)\n",
    "    )\n",
    "    return df\n",
    "\n",
    "# ─── Apply the derivation ────────────────────────────────────────\n",
    "df = build_targets(df)\n",
    "\n",
    "# Quick look at the result and type distribution (including co-occurring)\n",
    "print(\"\\nFailure type distribution:\")\n",
    "print(df[MULTI_CLASS_TARGET].value_counts().to_string())\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-7",
   "metadata": {},
   "source": [
    "## 1.4 — Data overview\n",
    "\n",
    "Finally we print a **data overview**: dimensions, types, missing values, and target distributions. This gives us a first picture that reveals:\n",
    "\n",
    "- The size of **class imbalance** — to be handled later in Phase 3.\n",
    "- Whether any values are missing from the start.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-8",
   "metadata": {},
   "outputs": [],
   "source": [
    "def data_overview(df: pd.DataFrame) -> None:\n",
    "    \"\"\"\n",
    "    Print a comprehensive summary of the data (shape, dtypes, missing values, target distributions).\n",
    "    \"\"\"\n",
    "    print(\"=\" * 55)\n",
    "    print(\"📊 Data Overview\")\n",
    "    print(\"=\" * 55)\n",
    "    print(f\"Rows    : {df.shape[0]:,}\")\n",
    "    print(f\"Columns : {df.shape[1]}\")\n",
    "\n",
    "    print(\"\\n── Column dtypes ──\")\n",
    "    print(df.dtypes.value_counts().to_string())\n",
    "\n",
    "    print(\"\\n── Missing values ──\")\n",
    "    missing = df.isnull().sum()\n",
    "    print(\"No missing values ✅\" if missing.sum() == 0 else missing[missing > 0])\n",
    "\n",
    "    print(\"\\n── Binary target distribution (Machine failure) ──\")\n",
    "    print(df[BINARY_TARGET].value_counts().to_string())\n",
    "    print(f\"Failure rate: {df[BINARY_TARGET].mean() * 100:.2f}%\")\n",
    "\n",
    "    print(\"\\n── Failure type distribution ──\")\n",
    "    print(df[MULTI_CLASS_TARGET].value_counts().to_string())\n",
    "\n",
    "# ─── Call the overview ───────────────────────────────────────\n",
    "data_overview(df)\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-9",
   "metadata": {},
   "source": [
    "## ✅ Phase 1 summary\n",
    "\n",
    "We accomplished the following:\n",
    "\n",
    "1. **Defined the two tasks** (binary and multi-class) and the goal of each.\n",
    "2. **Established a central constants structure** to be used in all later phases.\n",
    "3. **Wrote documented functions** for loading and building targets.\n",
    "4. **Discovered a documented internal inconsistency** between the failure flags and the `Machine failure` column, and adopted a clear, justified reconciliation policy.\n",
    "5. **Confirmed severe class imbalance** (failure rate ~3.4%) — this will drive our decisions in Phase 3 (Stratified K-Fold + `scale_pos_weight`).\n",
    "\n",
    "### 🔜 Next: Phase 2 — Exploratory Data Analysis (EDA)\n",
    "We will analyze the distribution of each sensor, plot the correlation matrix, and examine how sensor readings differ between healthy and failed machines.\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
